# Convert merged model to MLC for WebLLM (v5)

Standalone notebook: pulls the merged HF weights produced by `train.ipynb` (`virgilvox/conduyt-pilot-1.5b-v5-merged`), runs `mlc_llm convert_weight` + `mlc_llm gen_config` with `q4f16_1` quantization, and pushes to a new HF repo. Output is consumable by `@mlc-ai/web-llm` (browser inference, WebGPU).

Runs on Kaggle T4 (CPU-only MLC wheels work fine; conversion is arithmetic). Wall time is dominated by the upload at the end.

**v5 changes from v2:**
- `MERGED_REPO` and `MLC_REPO` use the v5 names so the v2 artifacts stay archived.
- WebLLM `model_lib` URL bumped to current `binary-mlc-llm-libs` layout: `web-llm-models/v0_2_83/base/Qwen2-1.5B-Instruct-q4f16_1_cs1k-webgpu.wasm`. The `ctx4k` segment was dropped from the wasm filename and a `base/` subfolder was introduced. HTTP-200 verified at notebook authoring time.
- CPU-only MLC wheels (`mlc-{llm,ai}-nightly-cpu`) unchanged from v2 (the cu128 nightlies in 2026-04 had a TVM regression; CPU works regardless of CUDA driver).


In [ ]:
# Install MLC nightly. Using the CPU-only wheels (mlc-{llm,ai}-nightly-cpu)
# instead of the cu* variants because:
#  - Weight conversion is arithmetic; GPU acceleration adds no value here.
#  - The cu128 / cu122 nightlies have been unstable in 2026-04 (TVM tirx
#    regressions, libcudart.so.12 link issues at import time).
#  - CPU wheels have no CUDA runtime dependency, so they import cleanly on
#    any Kaggle image regardless of CUDA toolkit version.
# Source: https://llm.mlc.ai/docs/install/mlc_llm.html
!pip install -q --upgrade pip
!pip install -q --pre -U -f https://mlc.ai/wheels \
    mlc-llm-nightly-cpu \
    mlc-ai-nightly-cpu
!pip install -q huggingface_hub
# Sanity-check the import works.
!python -c "import mlc_llm; print('mlc_llm at', mlc_llm.__file__)"


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")


In [ ]:
# Pull the merged HF weights produced by train.ipynb's final cell.
from huggingface_hub import snapshot_download

MERGED_REPO = "virgilvox/conduyt-pilot-1.5b-v5-merged"
MERGED_DIR  = "/kaggle/working/merged"

if not os.path.exists(MERGED_DIR):
    snapshot_download(
        repo_id   = MERGED_REPO,
        local_dir = MERGED_DIR,
        token     = os.environ["HF_TOKEN"],
    )
print("merged weights at", MERGED_DIR)
import os as _os
_os.system(f"du -sh {MERGED_DIR}")


In [ ]:
# Convert weights to q4f16_1 quantization. q4f16_1 is the standard WebLLM
# quantization for browser inference (4-bit weights, fp16 activations).
# --device cpu because we installed the CPU-only MLC wheels; conversion is
# arithmetic only, so the device choice is just about which TVM target gets
# JITed for the quantization kernels.
MLC_OUT = "/kaggle/working/mlc-q4f16_1"
!rm -rf {MLC_OUT}
!mkdir -p {MLC_OUT}

!python -m mlc_llm convert_weight \
    {MERGED_DIR} \
    --quantization q4f16_1 \
    --device cpu \
    --output {MLC_OUT}

# Sanity check: should see params_shard_*.bin files (~900MB total)
!ls -lh {MLC_OUT}/ | head -20
!du -sh {MLC_OUT}/


In [ ]:
# Generate the model config. --conv-template qwen2 covers Qwen2.5-Coder-1.5B-Instruct.
!python -m mlc_llm gen_config \
    {MERGED_DIR} \
    --quantization q4f16_1 \
    --conv-template qwen2 \
    --output {MLC_OUT}


In [ ]:
# Push to HF Hub.
from huggingface_hub import HfApi, create_repo

MLC_REPO = "virgilvox/conduyt-pilot-1.5b-v5-MLC"
api = HfApi(token=os.environ["HF_TOKEN"])
create_repo(MLC_REPO, exist_ok=True, token=os.environ["HF_TOKEN"])
api.upload_folder(
    folder_path = MLC_OUT,
    repo_id     = MLC_REPO,
    repo_type   = "model",
)
print("pushed to https://huggingface.co/" + MLC_REPO)


## WebLLM `appConfig` snippet

Drop the snippet below into your client-side WebLLM bootstrap. The `model` URL points at the v5 HF repo this notebook just uploaded; `model_lib` points at the prebuilt WASM library that matches Qwen2-1.5B-Instruct + q4f16_1.

The `model_lib` URL was confirmed against the current `mlc-ai/web-llm` `src/config.ts` (which references `web-llm-models/v0_2_83/base/`) and HTTP-200 verified.

Note: `Qwen2.5-Coder-1.5B-Instruct` and `Qwen2-1.5B-Instruct` share the same architecture (Qwen2 family, 1.5B params) so the same prebuilt WASM library handles both. Upstream WebLLM uses this same library for its own Qwen2.5-Coder entry.

```ts
import { CreateMLCEngine } from '@mlc-ai/web-llm'

const appConfig = {
  model_list: [
    {
      model_id:  'conduyt-pilot-1.5b-v5',
      model:     'https://huggingface.co/virgilvox/conduyt-pilot-1.5b-v5-MLC',
      model_lib: 'https://raw.githubusercontent.com/mlc-ai/binary-mlc-llm-libs/main/web-llm-models/v0_2_83/base/Qwen2-1.5B-Instruct-q4f16_1_cs1k-webgpu.wasm',
    },
  ],
}

const engine = await CreateMLCEngine('conduyt-pilot-1.5b-v5', { appConfig })
const reply  = await engine.chat.completions.create({
  messages: [
    { role: 'system', content: "You write conduyt-js host code only. JavaScript and TypeScript do not have digitalWrite, pinMode, millis, delay, or Arduino-style map." },
    { role: 'user',   content: 'Set up a servo on pin 9 with conduyt-js.' },
  ],
})
console.log(reply.choices[0].message.content)
```

If WebLLM ever moves the wasm path again (the directory has gone v0_2_30 -> v0_2_34 -> ... -> v0_2_80 -> v0_2_83 over time), check the current path with:

```
curl -sI https://raw.githubusercontent.com/mlc-ai/binary-mlc-llm-libs/main/web-llm-models/v0_2_83/base/Qwen2-1.5B-Instruct-q4f16_1_cs1k-webgpu.wasm
```

A 200 response means the URL is still current. A 404 means the libs directory has moved; re-derive from `mlc-ai/web-llm`'s `src/config.ts` on main.
